# 01 - Autenticacion

Primer contacto con la API v1 de IOL Inversiones a traves de `pyIol2`.

## Que se ve en este notebook

- Leer las credenciales del entorno.
- Crear el cliente `IOLClient` como context manager.
- Consultar `GET /health`, que no requiere token.
- Hacer la primera request autenticada.
- Usar la version `_raw()` de un metodo.

## Requisitos

- `uv sync --all-extras`
- `IOL_USERNAME` e `IOL_PASSWORD` exportados en el entorno.


## 1. Credenciales

Se leen de `os.environ`. El placeholder esta marcado con `# completar`.

In [ ]:
import os

# completar: exportar las credenciales en el entorno antes de ejecutar.
#   export IOL_USERNAME=tu_usuario_iol
#   export IOL_PASSWORD=tu_password_iol
# Nunca escribir las credenciales en el notebook.
USERNAME = os.environ.get("IOL_USERNAME", "")  # completar
PASSWORD = os.environ.get("IOL_PASSWORD", "")  # completar

if not USERNAME or not PASSWORD:
    raise SystemExit("Faltan IOL_USERNAME e IOL_PASSWORD en el entorno")


## 2. Constantes

Los valores de protocolo salen de `constants.py`: no se escriben como literales.

In [ ]:
from pyIol2 import Countries, Markets, SettlementTerms, __version__

print("pyIol2", __version__)
print("mercado por defecto:", Markets.BCBA)
print("plazo por defecto:", SettlementTerms.T0)
print("pais por defecto:", Countries.ARGENTINA)


## 3. Cliente

El `with` cierra el `httpx.Client` al salir. La autenticacion es perezosa: el token se pide en la primera request autenticada.

In [ ]:
from pyIol2 import IOLClient

client = IOLClient(USERNAME, PASSWORD)
print("base_url:", client.base_url)
print("timeout:", client.timeout)


## 4. Estado del servicio

`get_health_status()` es el unico metodo sin token: separa un problema de IOL de un problema de credenciales.

In [ ]:
from pyIol2 import IOLAPIError

try:
    estado = client.get_health_status()
    print("status:", estado.status)
    print("version:", estado.version)
    print("uptime:", estado.uptime)
except IOLAPIError as exc:
    print("la API no respondio:", exc)


## 5. Primera request autenticada

Si las credenciales son validas, los saldos responden. El `trace_id` aparece en las excepciones.

In [ ]:
try:
    saldos = client.get_account_balances(Countries.ARGENTINA)
    print("ARS disponible T0:", saldos.ars.available.t0)
    print("USD disponible T0:", saldos.usd.available.t0)
except IOLAPIError as exc:
    print("status:", exc.status_code)
    print("mensaje:", exc.message)
    print("trace_id:", exc.trace_id)
    raise


## 6. Version `_raw()`

Devuelve el campo `data` de la respuesta sin transformar. No incluye `meta` ni `error`.

In [ ]:
try:
    crudo = client.get_health_status_raw()
    print(sorted(crudo.keys()))
except IOLAPIError as exc:
    print("fallo:", exc)


## 7. Cierre

Cerrar el cliente si no se uso como context manager.

In [ ]:
client.close()
print("cliente cerrado")
